# Hedging Against Grid Uncertainty with Two-Stage Stochastic Optimization in PyPSA

This notebook investigates how an electricity system **hedges against uncertainty in grid (transmission) capacity** using PyPSA's native stochastic optimization (PyPSA ≥ 1.0).

**Setup:**
- 5-node electricity-only network with solar, wind, gas, coal and battery storage (all extendable)
- Two grid realizations: a **low grid** (30% of base line capacity, e.g. delayed grid expansion) and a **high grid** (150%, e.g. successful expansion)
- Wind is only available in the "north" (buses 3, 4), far from the load centers in the "south" — so grid capacity determines how much remote wind is actually usable

**Experiments:**
1. Deterministic optimization of the *low grid* network
2. Deterministic optimization of the *high grid* network
3. **Two-stage stochastic optimization** over both grid scenarios (50/50): capacities are *here-and-now* first-stage decisions, dispatch is a scenario-specific *wait-and-see* recourse

Finally we compare optimal capacities and system costs, and compute EVPI and ECIU/VSS.

References: [Stochastic optimization user guide](https://docs.pypsa.org/v1.0.0/user-guide/optimization/stochastic/), [legacy stochastic example](https://docs.pypsa.org/v0.35.0/examples/stochastic-problem.html)

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyproj
pyproj.datadir.set_data_dir("/home/julian-geis/repos/pypsa-de/.pixi/envs/default/share/proj")
import pypsa

warnings.filterwarnings("ignore", category=FutureWarning)

import logging
for logger in ["pypsa", "linopy"]:
    logging.getLogger(logger).setLevel(logging.ERROR)
plt.rcParams["figure.dpi"] = 110

print("PyPSA version:", pypsa.__version__)

## Model settings

We use a 72-hour horizon (3 days) with synthetic load and renewable profiles. Annualized capital costs are scaled to the horizon length so that CAPEX/OPEX trade-offs remain realistic.

In [ ]:
rng = np.random.default_rng(42)

SNAPSHOTS = pd.date_range("2026-01-01", periods=72, freq="h")
t = np.arange(len(SNAPSHOTS))
ANNUITY = len(SNAPSHOTS) / 8760  # scale annualized capital costs to the horizon

BASE_S_NOM = 1200.0  # MW per line
GRID_FACTORS = {"low_grid": 0.3, "high_grid": 1.5}
PROBABILITIES = {"low_grid": 0.5, "high_grid": 0.5}

VOLL = 3000.0  # value of lost load, EUR/MWh

# common exogenous time series (identical across scenarios -> only the grid is uncertain)
DAILY = 1 + 0.35 * np.sin(2 * np.pi * (t - 8) / 24)
LOAD_SCALE = {"bus0": 1500, "bus1": 2500, "bus2": 1800, "bus3": 300, "bus4": 400}  # MW
LOAD_PROFILES = {
    b: s * DAILY * (1 + 0.05 * rng.standard_normal(len(t))) for b, s in LOAD_SCALE.items()
}
SOLAR_PU = np.clip(np.sin(np.pi * ((t % 24) - 6) / 12), 0, None)
WIND_PU = np.clip(
    0.4 + 0.3 * np.sin(2 * np.pi * t / 36) + 0.15 * rng.standard_normal(len(t)), 0.02, 1.0
)
RES_SCALE = {name: 0.85 + 0.15 * rng.random() for name in ["wind_bus3", "wind_bus4", "solar_bus0", "solar_bus2"]}

## Network builder

All generation and storage capacities are extendable (**first-stage** variables). Lines are **not** extendable — their `s_nom` is the uncertain parameter that differs between scenarios. A load-shedding generator at every bus guarantees feasibility.

Topology: a ring `bus0–bus1–bus2–bus4–bus3–bus0` plus a chord `bus1–bus4`. Load sits in the south (buses 0–2), wind potential in the north (buses 3–4).

In [ ]:
def build_network(grid_factor=1.0):
    """5-node electricity-only capacity expansion model with fixed line capacities."""
    n = pypsa.Network()
    n.set_snapshots(SNAPSHOTS)

    colors = {"solar": "gold", "wind": "steelblue", "gas": "indianred",
              "coal": "dimgray", "battery": "seagreen", "AC": "black",
              "load shedding": "magenta"}
    for c, color in colors.items():
        n.add("Carrier", c, color=color)

    coords = {"bus0": (0, 0), "bus1": (2, 0), "bus2": (4, 0), "bus3": (1, 2), "bus4": (3, 2)}
    for b, (x, y) in coords.items():
        n.add("Bus", b, x=x, y=y, carrier="AC")

    edges = [("bus0", "bus1"), ("bus1", "bus2"), ("bus2", "bus4"),
             ("bus4", "bus3"), ("bus3", "bus0"), ("bus1", "bus4")]
    for i, (b0, b1) in enumerate(edges):
        n.add("Line", f"line{i}", bus0=b0, bus1=b1, x=0.1, r=0.01,
              s_nom=BASE_S_NOM * grid_factor, s_nom_extendable=False, carrier="AC")

    for b, profile in LOAD_PROFILES.items():
        n.add("Load", f"load_{b}", bus=b, p_set=profile)

    # wind only in the north, far from load
    for b in ["bus3", "bus4"]:
        n.add("Generator", f"wind_{b}", bus=b, carrier="wind", p_nom_extendable=True,
              p_max_pu=WIND_PU * RES_SCALE[f"wind_{b}"],
              capital_cost=90_000 * ANNUITY, marginal_cost=0.02)
    # solar in the south, close to load
    for b in ["bus0", "bus2"]:
        n.add("Generator", f"solar_{b}", bus=b, carrier="solar", p_nom_extendable=True,
              p_max_pu=SOLAR_PU * RES_SCALE[f"solar_{b}"],
              capital_cost=50_000 * ANNUITY, marginal_cost=0.01)
    # gas peakers at the load centers
    for b in ["bus0", "bus1", "bus2"]:
        n.add("Generator", f"gas_{b}", bus=b, carrier="gas", p_nom_extendable=True,
              efficiency=0.55, capital_cost=47_000 * ANNUITY, marginal_cost=70)
    # coal baseload option at the largest load center
    n.add("Generator", "coal_bus1", bus="bus1", carrier="coal", p_nom_extendable=True,
          efficiency=0.4, capital_cost=120_000 * ANNUITY, marginal_cost=32)
    # batteries and load shedding everywhere
    for b in coords:
        n.add("StorageUnit", f"battery_{b}", bus=b, carrier="battery", p_nom_extendable=True,
              max_hours=4, efficiency_store=0.95, efficiency_dispatch=0.95,
              capital_cost=60_000 * ANNUITY, cyclic_state_of_charge=True)
        n.add("Generator", f"shed_{b}", bus=b, carrier="load shedding",
              p_nom=1e5, marginal_cost=VOLL)
    return n

In [ ]:
# quick look at the topology
n_plot = build_network()
fig, ax = plt.subplots(figsize=(7, 4))
for _, line in n_plot.lines.iterrows():
    x = [n_plot.buses.at[line.bus0, "x"], n_plot.buses.at[line.bus1, "x"]]
    y = [n_plot.buses.at[line.bus0, "y"], n_plot.buses.at[line.bus1, "y"]]
    ax.plot(x, y, color="gray", lw=2, zorder=1)
ax.scatter(n_plot.buses.x, n_plot.buses.y, s=[LOAD_SCALE[b] / 4 for b in n_plot.buses.index],
           color="firebrick", zorder=2, label="load (size)")
for b, row in n_plot.buses.iterrows():
    ax.annotate(b, (row.x, row.y), textcoords="offset points", xytext=(8, 8))
ax.annotate("wind potential", (2, 2.25), ha="center", color="steelblue", fontsize=11)
ax.annotate("solar potential + load centers", (2, -0.45), ha="center", color="darkgoldenrod", fontsize=11)
ax.set_title("5-node test network (uncertain line capacities)")
ax.set_axis_off()
ax.legend(loc="center right")
plt.show()

## 1) + 2) Deterministic optimization of the low- and high-grid networks

These are the *wait-and-see* solutions: each assumes perfect knowledge of the grid realization before investing.

In [ ]:
deterministic = {}
for name, factor in GRID_FACTORS.items():
    n = build_network(factor)
    status, condition = n.optimize(solver_name="highs", log_to_console=False)
    assert condition == "optimal"
    deterministic[name] = n
    print(f"{name:>10}: objective = {n.objective/1e6:8.2f} M EUR")

## 3) Stochastic optimization over both grid topologies

We build the network once, call `n.set_scenarios()`, and then overwrite the line capacities per scenario. Investment variables (`p_nom`) remain scenario-independent first-stage decisions (non-anticipativity), while dispatch becomes scenario-indexed.

In [ ]:
GRID_FACTORS

In [ ]:
n_stoch = build_network()
n_stoch.set_scenarios(PROBABILITIES)

# scenario-specific line capacities (the uncertain parameter)
for scen, factor in GRID_FACTORS.items():
    n_stoch.lines.loc[(scen, slice(None)), "s_nom"] = BASE_S_NOM * factor

n_stoch.lines[["s_nom"]].unstack("scenario")

In [ ]:
status, condition = n_stoch.optimize(solver_name="highs", log_to_console=False)
assert condition == "optimal"
print(f"stochastic: expected objective = {n_stoch.objective/1e6:8.2f} M EUR")

In [ ]:
# first-stage capacities are identical across scenarios (non-anticipativity) -> these are the optimised capacities for the stochstic scenario
n_stoch.statistics.optimal_capacity().unstack("scenario").round(0)

## Comparison: optimal capacities

We collect the installed capacities by carrier for all three runs. The stochastic solution must find a *single* capacity portfolio that works under both grid realizations.

In [ ]:
def capacities_by_carrier(n):
    caps = []
    gens = n.generators
    sus = n.storage_units
    if isinstance(gens.index, pd.MultiIndex):  # stochastic: take one scenario (first stage is identical)
        gens = gens.xs(n.scenarios[0], level="scenario")
        sus = sus.xs(n.scenarios[0], level="scenario")
    caps.append(gens.groupby("carrier").p_nom_opt.sum())
    caps.append(sus.groupby("carrier").p_nom_opt.sum())
    s = pd.concat(caps).drop("load shedding", errors="ignore")
    return s

caps = pd.DataFrame({
    "deterministic low grid": capacities_by_carrier(deterministic["low_grid"]),
    "deterministic high grid": capacities_by_carrier(deterministic["high_grid"]),
    "stochastic (50/50)": capacities_by_carrier(n_stoch),
}).fillna(0)
caps.round(0)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
caps.div(1e3).plot.bar(ax=ax, color=["#c6dbef", "#2171b5", "#e6550d"], width=0.75)
ax.set_ylabel("installed capacity [GW]")
ax.set_xlabel("")
ax.set_title("Optimal capacities: deterministic vs. stochastic")
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Comparison: system costs

For each capacity portfolio we ask: *what would it cost under each grid realization?* For the two deterministic portfolios this requires evaluating them under the other scenario as well — that is exactly what the ECIU/VSS metric captures. Here we first compare the direct objectives, then compute the standard stochastic-programming metrics:

- **WS** (wait-and-see): expected cost if the grid outcome were known before investing, $\mathbb{E}[\mathrm{WS}] = \sum_s p_s \, \mathrm{obj}_s^{\det}$
- **SP**: the stochastic solution's expected cost
- **EEV**: take naive planner who ingores uncertainty by replacing the random parameters with its expected value (here 0.5⋅0.3⋅1200+0.5⋅1.5⋅1200=0.9⋅1200=1080 MW for the line capacities) -> solve capacitites -> solve dispatch for both independently and receive EEV = capex + probability wieghted opex of scenario realizations
- **EVPI** $= \mathrm{SP} - \mathbb{E}[\mathrm{WS}]$: the value of perfect information about the grid
- **EEV / ECIU**: expected cost of naively deploying the portfolio optimized for the *expected* grid, evaluated under both scenarios; $\mathrm{ECIU} = \mathrm{EEV} - \mathrm{SP}$ is the value of stochastic planning

In [ ]:
def evaluate_portfolio(source_n, grid_factor):
    """Fix the optimal capacities of `source_n` and evaluate dispatch under a given grid."""
    m = build_network(grid_factor)
    gens = source_n.generators
    sus = source_n.storage_units
    if isinstance(gens.index, pd.MultiIndex):
        gens = gens.xs(source_n.scenarios[0], level="scenario")
        sus = sus.xs(source_n.scenarios[0], level="scenario")
    m.generators.loc[gens.index, "p_nom"] = gens.p_nom_opt
    m.storage_units.loc[sus.index, "p_nom"] = sus.p_nom_opt
    m.generators.p_nom_extendable = False
    m.storage_units.p_nom_extendable = False
    status, condition = m.optimize(solver_name="highs", log_to_console=False)
    assert condition == "optimal"
    capex = (m.generators.capital_cost * m.generators.p_nom).sum() + \
            (m.storage_units.capital_cost * m.storage_units.p_nom).sum()
    return capex + m.objective  # objective of dispatch-only run = OPEX

# cost of each portfolio under each grid realization
portfolios = {
    "deterministic low grid": deterministic["low_grid"],
    "deterministic high grid": deterministic["high_grid"],
    "stochastic (50/50)": n_stoch,
}
cost_matrix = pd.DataFrame({
    pname: {scen: evaluate_portfolio(pn, f) for scen, f in GRID_FACTORS.items()}
    for pname, pn in portfolios.items()
}).T
cost_matrix["expected"] = sum(PROBABILITIES[s] * cost_matrix[s] for s in PROBABILITIES)
(cost_matrix / 1e6).round(2)

- rows are the portfolios / networks and columns are the realizations
- determinisitc low grid evaluated on low grid has the least costs as it was optimized on it -> 18.82 (same for 17.22)
- High-grid portfolio in the low-grid world: 115.64. Catastrophic regret. The portfolio bet on remote wind reaching the load centers; behind a 0.3× grid that wind is stranded, the load centers have too little local capacity, and the gap is filled by load shedding at 3000 €/MWh VOLL.
- The stochastic row (19.06 / 18.14) sits strictly between the deterministic extremes in both columns:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [1.4, 1]})

(cost_matrix[["low_grid", "high_grid"]] / 1e6).plot.bar(
    ax=axes[0], color=["#e6550d", "#2171b5"], width=0.7)
axes[0].set_ylabel("total system cost [M EUR]")
axes[0].set_title("Portfolio cost by grid realization")
axes[0].tick_params(axis="x", rotation=15)
axes[0].grid(axis="y", alpha=0.3)

(cost_matrix["expected"] / 1e6).plot.bar(ax=axes[1], color=["#c6dbef", "#2171b5", "#e6550d"], width=0.6)
ws = sum(PROBABILITIES[s] * deterministic[s].objective for s in PROBABILITIES)
axes[1].axhline(ws / 1e6, color="k", ls="--", lw=1, label="E[WS] (perfect information)")
axes[1].set_title("Expected system cost")
axes[1].set_ylabel("expected cost [M EUR]")
axes[1].tick_params(axis="x", rotation=15)
axes[1].grid(axis="y", alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()

# plot of costs from above matrix

In [ ]:
# stochastic-programming metrics
SP = n_stoch.objective
WS = sum(PROBABILITIES[s] * deterministic[s].objective for s in PROBABILITIES)
EVPI = SP - WS

# EEV: portfolio optimized for the *expected* grid capacity, evaluated under both scenarios
mean_factor = sum(PROBABILITIES[s] * f for s, f in GRID_FACTORS.items())
n_ev = build_network(mean_factor)
n_ev.optimize(solver_name="highs", log_to_console=False)
EEV = sum(PROBABILITIES[s] * evaluate_portfolio(n_ev, f) for s, f in GRID_FACTORS.items())
ECIU = EEV - SP

print(f"E[WS]  (perfect information)      = {WS/1e6:7.2f} M EUR") # (18.82 + 17.22)/2
print(f"SP     (stochastic solution)      = {SP/1e6:7.2f} M EUR")
print(f"EEV    (expected-value solution)  = {EEV/1e6:7.2f} M EUR")
print(f"EVPI = SP - E[WS]                 = {EVPI/1e6:7.2f} M EUR ({EVPI/SP:.1%} of SP)")
print(f"ECIU = EEV - SP                   = {ECIU/1e6:7.2f} M EUR ({ECIU/SP:.1%} of SP)")
assert WS <= SP + 1e-3 and SP <= EEV + 1e-3  # WS <= SP <= EEV

In [ ]:
def evaluate_portfolio_detailed(source_n, grid_factor):
    """Fix optimal capacities of `source_n`, run dispatch-only under given grid.
    Returns (capex, opex) separately."""
    m = build_network(grid_factor)
    gens = source_n.generators
    sus = source_n.storage_units
    if isinstance(gens.index, pd.MultiIndex):
        gens = gens.xs(source_n.scenarios[0], level="scenario")
        sus = sus.xs(source_n.scenarios[0], level="scenario")
    m.generators.loc[gens.index, "p_nom"] = gens.p_nom_opt
    m.storage_units.loc[sus.index, "p_nom"] = sus.p_nom_opt
    m.generators.p_nom_extendable = False
    m.storage_units.p_nom_extendable = False
    status, condition = m.optimize(solver_name="highs", log_to_console=False)
    assert condition == "optimal"
    capex = (m.generators.capital_cost * m.generators.p_nom).sum() + \
            (m.storage_units.capital_cost * m.storage_units.p_nom).sum()
    opex = m.objective  # dispatch-only objective = pure OPEX (incl. load shedding)
    return capex, opex


# --- Step 1: EV problem (naive planner optimizes on the fictitious mean grid) ---
mean_factor = sum(PROBABILITIES[s] * f for s, f in GRID_FACTORS.items())
n_ev = build_network(mean_factor)
n_ev.optimize(solver_name="highs", log_to_console=False)
print(f"EV problem: capacities optimized on mean grid "
      f"(factor {mean_factor:.2f} = {BASE_S_NOM * mean_factor:.0f} MW/line)")
print(f"EV objective (fictitious world, NOT the EEV) = {n_ev.objective/1e6:8.2f} M EUR\n")

# --- Steps 2+3: freeze capacities, dispatch under each real scenario ---
capex = None
opex = {}
for scen, f in GRID_FACTORS.items():
    capex, opex[scen] = evaluate_portfolio_detailed(n_ev, f)

expected_opex = sum(PROBABILITIES[s] * opex[s] for s in PROBABILITIES)
EEV = capex + expected_opex

print(f"{'CAPEX (frozen EV portfolio, counted once)':<45} = {capex/1e6:8.2f} M EUR")
for scen in GRID_FACTORS:
    print(f"{'OPEX dispatch in ' + scen:<45} = {opex[scen]/1e6:8.2f} M EUR"
          f"  (p = {PROBABILITIES[scen]})")
print(f"{'Expected OPEX = sum_s p_s * opex_s':<45} = {expected_opex/1e6:8.2f} M EUR")
print("-" * 68)
print(f"{'EEV = CAPEX + expected OPEX':<45} = {EEV/1e6:8.2f} M EUR")
print(f"{'SP  (stochastic solution)':<45} = {SP/1e6:8.2f} M EUR")
print(f"{'ECIU = EEV - SP':<45} = {(EEV - SP)/1e6:8.2f} M EUR")


## How does the hedge operate? Scenario-specific dispatch

The first-stage capacities are shared, but dispatch adapts to the realized grid. In the low-grid scenario the north wind is partly stranded behind congested lines, so more gas/coal is dispatched at the load centers.

In [ ]:
eb = (n_stoch.statistics.energy_balance()
      .unstack("scenario")
      .groupby("carrier").sum()
      .drop(["-", "AC"], errors="ignore"))
supply = eb.clip(lower=0).div(1e3)
supply = supply[supply.sum(axis=1) > 1e-6]

fig, ax = plt.subplots(figsize=(7, 4))
supply.T.plot.bar(ax=ax, stacked=True, width=0.5,
                  color=[n_plot.carriers.color.get(c, "gray") for c in supply.index])
ax.set_ylabel("electricity supplied [GWh]")
ax.set_title("Stochastic solution: energy mix by scenario (wait-and-see dispatch)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=0)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def supply_by_carrier(n, scenario=None):
    """Positive side of the energy balance, grouped by carrier [GWh]."""
    eb = n.statistics.energy_balance()
    if scenario is not None:
        eb = eb.xs(scenario, level="scenario")
    s = (eb.groupby("carrier").sum()
           .drop(["-", "AC"], errors="ignore")
           .clip(lower=0)
           .div(1e3))
    return s[s > 1e-6]

supply = pd.DataFrame({
    "det. low grid": supply_by_carrier(deterministic["low_grid"]),
    "det. high grid": supply_by_carrier(deterministic["high_grid"]),
    "stochastic | low grid": supply_by_carrier(n_stoch, "low_grid"),
    "stochastic | high grid": supply_by_carrier(n_stoch, "high_grid"),
}).fillna(0)

# consistent carrier order and colors
order = [c for c in ["coal", "gas", "solar", "wind", "battery", "load shedding"]
         if c in supply.index]
supply = supply.loc[order]

fig, ax = plt.subplots(figsize=(9, 4.5))
supply.T.plot.bar(ax=ax, stacked=True, width=0.6,
                  color=[n_plot.carriers.color.get(c, "gray") for c in supply.index])
ax.set_ylabel("electricity supplied [GWh]")
ax.set_xlabel("")
ax.set_title("Energy mix: deterministic portfolios vs. stochastic wait-and-see dispatch")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=15, ha="right")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Takeaways

- **Deterministic high grid** builds the most (remote) wind, since ample transmission delivers it to the southern load centers, and the least storage — the cheapest system overall.
- **Deterministic low grid** substitutes wind with local solar, batteries, gas and coal at the load centers, at markedly higher cost.
- The **stochastic solution hedges**: it commits to a portfolio closer to the low-grid one (less wind, more solar/batteries and some peaking gas), accepting slight over-investment if the grid turns out strong in exchange for avoiding severe congestion costs if it turns out weak. The costly, asymmetric downside of the low-grid scenario dominates the hedge.
- Each deterministic portfolio performs poorly when the *other* grid realization occurs — the wind-heavy high-grid portfolio is especially expensive under the weak grid (stranded wind, load served by expensive recourse).
- **EVPI** quantifies the value of resolving grid uncertainty (e.g. regulatory certainty about grid expansion) before investing; **ECIU/VSS** quantifies the value of stochastic planning over naive expected-value planning. Its large magnitude here is driven by load shedding at VOLL: the expected-value portfolio (sized for the mean grid) leaves demand unserved behind congested lines once the weak grid materializes.

**Extensions to explore:** asymmetric probabilities (e.g. 80% chance of grid delay), more grid scenarios, CVaR risk aversion via `n.set_risk_preference(alpha=..., omega=...)`, or making lines extendable in the first stage to co-optimize grid vs. generation hedging.